# Semiconductor Device QC — GPU training on Google Colab

Trains the **large profile**: **YOLO11s-seg** defect segmenter on **384 × 384** device crops for **100 epochs**
(the laptop/CPU version uses YOLO11n-seg, 256 px, 25 epochs). Then evaluates the full pipeline on the
held-out test split, exactly like `python run_project.py evaluate`.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Total time ≈ 1–1.5 h on a free T4.

Author: Balasudhan R

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/BALASUDHAN1807/WaferSense-AI.git
%cd WaferSense-AI
!pip -q install ultralytics onnx onnxruntime scikit-image

## 1. Settings: large profile on the GPU

In [ ]:
import os
os.environ["QC_PROFILE"] = "large"   # YOLO11s-seg, 384 px crops, 100 epochs
os.environ["QC_DEVICE"] = "0"        # first GPU
os.environ["PYTHONPATH"] = "src"
!python -c "import sys; sys.path.insert(0,'src'); from qc import config as C; print(C.PROFILE, C.DEVICE, C.CROP_SIZE, C.YOLO_SEG)"

## 2. Generate the synthetic dataset (same seed as the CPU version → identical frames, 384 px crops)

In [ ]:
!python run_project.py generate
!python run_project.py validate

## 3. Train YOLO11s-seg for 100 epochs (~1 h on a T4)

In [ ]:
!python run_project.py train-yoloseg

## 4. Evaluate the complete pipeline on the held-out test split

In [ ]:
!python run_project.py evaluate
import json
ev = json.load(open("results/large/reports/evaluation.json"))
e = ev["end_to_end"]["yolo"]
print("Decision accuracy:", round(e["decision_accuracy"], 4), " macro F1:", round(e["decision_macro_f1"], 4))
print("Mean defect IoU:", round(ev["pixel"]["yolo"]["mean_IoU"], 4))
print({c: v["recall"] for c, v in e["defect_type_recognition"].items()})

## 5. Download the trained model + results

Then put the zip in your **Downloads** folder and tell Claude — it will add the model to the project, compare it with the CPU model and update the report and the Vercel demo.

In [ ]:
!zip -r large_profile_results.zip models/segmenter_yolo11s_seg_384.pt results/large/reports results/large/graphs results/large/predictions
from google.colab import files
files.download("large_profile_results.zip")

## (Optional) 6. Real images: MVTec AD transistor on the GPU

Upload `transistor.tar.xz` (from https://www.mvtec.com/company/research/datasets/mvtec-ad/downloads) with the
file browser on the left into `WaferSense-AI/dataset/real/`, then run:

In [ ]:
os.environ["QC_PROFILE"] = "base"
!python run_project.py real-eval
!zip -r real_results.zip results/reports/real_*.json results/graphs/real_* models/patchcore_transistor.pt
files.download("real_results.zip")